# Create the workspace bucket, run this only once

In [ ]:
%run 00_creating_workspace_bucket.ipynb

# Set environment varialbes

In [ ]:
%run 00_Setting_Env_Variables.ipynb

# Run analysis

## load functions used in the analysis

In [ ]:
%run 01_functions.ipynb

## Extracting non-genetic data and epidemiology analysis

In [ ]:
%run 02_cohort_data_collection_analysis.ipynb

## Create R environment and install dependency

In [ ]:
import subprocess

subprocess.run(
    [
        "bash",
        "create_r_env.sh",
        "r453",
        "4.5.3",
        "ir-r453",
        "R [conda env:r453]",
    ],
    check=True,
)

## Clinical associations

In [ ]:
import os
import papermill as pm

excute_notebook = pm.execute_notebook(
    input_path="03_association_analysis.ipynb",
    output_path="03_association_analysis_executed.ipynb",
    kernel_name="ir-r453",
    progress_bar=True,
    log_output=False,
)

## Merging phenotype data with PCs

In [ ]:
%run 04_01_merge_phenotype_PGS.ipynb

## Extracting whole genome sequencing data, data preprocessing and quality control

In [ ]:
%run 04_02_genotype_data_extraction_and_preprocessing.ipynb

## Polygenic score calculation using PRS-CS


In [ ]:
%run 05_PGS_PRScs.ipynb

## Integrated clinical and multi-trait polygenic modeling of complex polypharmacy

In [ ]:
excute_notebook = pm.execute_notebook(
    input_path="06_multi_clinical_PGSs_predict_polypharmacy.ipynb",
    output_path="06_multi_clinical_PGSs_predict_polypharmacy_executed.ipynb",
    kernel_name="ir-r453",
    progress_bar=True,
    log_output=False,
)

## Associations between individual PGS and polypharmacy 

In [ ]:
excute_notebook = pm.execute_notebook(
    input_path="07_individual_PGS_polypharmacy_associations.ipynb",
    output_path="07_individual_PGS_polypharmacy_associations_executed.ipynb",
    kernel_name="ir-r453",
    progress_bar=True,
    log_output=False
)

## Copy results to workspace bucket 

In [ ]:
import os
import subprocess
from pathlib import Path

src = Path("/home/jupyter/workspace")
bucket = os.environ["WORKSPACE_BUCKET"]
dest = f"{bucket}/results/tables"

xlsx_files = list(src.glob("*.xlsx"))

if not xlsx_files:
    print("No .xlsx files found.")
else:
    subprocess.run(
        ["gcloud", "storage", "mv", *map(str, xlsx_files), dest],
        check=True,
    )
    print(f"Copied {len(xlsx_files)} files to {dest}")